# Advanced Mutual Fund Analytics & Risk Metrics

This notebook covers risk, behavior, and structural portfolio analytics for 40 mutual fund schemes using historical NAV data, holdings snapshots, and transaction records from the `bluestock_mf.db` database.

## Notebook Outline:
1. **Historical Value-at-Risk (VaR 95%) and Conditional Value-at-Risk (CVaR)** across all 40 schemes.
2. **Rolling 90-Day Sharpe Ratio** analysis and visualization for 5 key funds.
3. **Investor Cohort Analysis** grouped by first transaction year.
4. **SIP Continuity and Churn Risk Analysis**.
5. **Simple Fund Recommender Engine** using risk appetites.
6. **Sector Concentration Analysis (Herfindahl-Hirschman Index - HHI)** for equity portfolios.
7. **Advanced Insights & Strategic Takeaways**.

In [1]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Connect to the database
db_path = r"../Day 2 - Cleaned data + SQLite DB loaded/bluestock_mf.db"
conn = sqlite3.connect(db_path)

# Load core datasets
df_funds = pd.read_sql_query("SELECT * FROM dim_fund;", conn)
df_nav_raw = pd.read_sql_query("SELECT * FROM fact_nav;", conn)
df_bench_raw = pd.read_sql_query("SELECT * FROM fact_benchmark_indices;", conn)
df_tx = pd.read_sql_query("SELECT * FROM fact_transactions;", conn)
df_holdings = pd.read_sql_query("SELECT * FROM fact_portfolio_holdings;", conn)
df_performance = pd.read_sql_query("SELECT * FROM fact_performance;", conn)

# Align NAV data to actual trading days (dates present in benchmark table)
trading_dates = sorted(df_bench_raw['date'].unique())
df_nav = df_nav_raw[df_nav_raw['date'].isin(trading_dates)].copy()
df_nav = df_nav.sort_values(['amfi_code', 'date'])
df_nav['date'] = pd.to_datetime(df_nav['date'])
df_nav.set_index('date', inplace=True)

print(f"Loaded {len(df_funds)} funds, {len(df_nav)} trading-day NAV records, and {len(df_tx)} transaction logs.")

## 1. Historical Value-at-Risk (VaR 95%) & Conditional Value-at-Risk (CVaR)

- **95% Historical VaR**: Represents the 5th percentile of daily returns (the threshold below which daily returns fall only 5% of the time).
- **95% CVaR (Expected Shortfall)**: The mean of returns below the 95% VaR threshold, representing the average return on tail-risk days.

In [2]:
# Compute daily returns
df_nav['daily_return'] = df_nav.groupby('amfi_code')['nav'].pct_change()

var_cvar_data = []
for amfi in df_funds['amfi_code']:
    fund_name = df_funds[df_funds['amfi_code'] == amfi]['scheme_name'].values[0]
    returns = df_nav[df_nav['amfi_code'] == amfi]['daily_return'].dropna()
    if len(returns) > 0:
        var_95 = returns.quantile(0.05)
        cvar_95 = returns[returns <= var_95].mean()
        var_cvar_data.append({
            'amfi_code': amfi,
            'scheme_name': fund_name,
            'var_95_pct': var_95 * 100,
            'cvar_95_pct': cvar_95 * 100
        })
df_var_cvar = pd.DataFrame(var_cvar_data)
df_var_cvar.to_csv('var_cvar_report.csv', index=False)

print("Top 5 Highest Risk Funds (Most negative 95% VaR):")
display(df_var_cvar.sort_values('var_95_pct', ascending=True).head(5))

print("\nTop 5 Lowest Risk Funds (Least negative 95% VaR):")
display(df_var_cvar.sort_values('var_95_pct', ascending=False).head(5))

## 2. Rolling 90-Day Sharpe Ratio Over Time

We calculate the rolling 90-day Sharpe ratio for 5 key equity schemes to see how their risk-adjusted return evolves over time:
$$\text{Sharpe Ratio}_{\text{rolling}} = \frac{\mu_{\text{returns}}}{\sigma_{\text{returns}}} \times \sqrt{252}$$

In [3]:
key_amfi = [148567, 120505, 120843, 100033, 120504]
colors = ['#1E3A8A', '#0D9488', '#8B5CF6', '#F59E0B', '#EF4444']

plt.figure(figsize=(14, 8))
for i, amfi in enumerate(key_amfi):
    fund_name = df_funds[df_funds['amfi_code'] == amfi]['scheme_name'].values[0]
    clean_name = fund_name.replace(" - Regular - Growth", "").replace(" - Direct - Growth", "").replace(" Plan - Growth", "")
    returns = df_nav[df_nav['amfi_code'] == amfi]['daily_return'].dropna()
    
    rolling_mean = returns.rolling(90).mean()
    rolling_std = returns.rolling(90).std()
    rolling_sharpe = (rolling_mean / rolling_std) * np.sqrt(252)
    
    plt.plot(rolling_sharpe.index, rolling_sharpe, label=clean_name, color=colors[i], linewidth=2.0)

plt.title("Rolling 90-Day Sharpe Ratio Over Time", fontsize=16, fontweight='bold', pad=15)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Annualized Sharpe Ratio (Rolling 90-Day)", fontsize=12)
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.legend(loc='upper right', frameon=True, framealpha=0.9)
plt.tight_layout()
plt.savefig('rolling_sharpe_chart.png', dpi=300)
plt.show()

## 3. Investor Cohort Analysis

We group investors by the calendar year of their very first transaction. For each cohort, we calculate:
- Average SIP Amount (SIP transaction type only)
- Total Invested Amount (SIP + Lumpsum transactions)
- Top Fund Preference (The mutual fund scheme with the largest total invested volume in INR)

In [4]:
df_tx['transaction_date'] = pd.to_datetime(df_tx['transaction_date'])
first_tx = df_tx.groupby('investor_id')['transaction_date'].min().reset_index(name='first_tx_date')
first_tx['cohort_year'] = first_tx['first_tx_date'].dt.year
df_tx_cohort = df_tx.merge(first_tx[['investor_id', 'cohort_year']], on='investor_id')

cohort_res = []
for cohort, group in df_tx_cohort.groupby('cohort_year'):
    # Total invested (SIP + Lumpsum)
    invested_group = group[group['transaction_type'].isin(['SIP', 'Lumpsum'])]
    total_invested = invested_group['amount_inr'].sum()
    
    # Average SIP amount
    sip_group = group[group['transaction_type'] == 'SIP']
    avg_sip = sip_group['amount_inr'].mean()
    
    # Top fund preference
    fund_pref = invested_group.groupby('amfi_code')['amount_inr'].sum().reset_index()
    fund_pref = fund_pref.merge(df_funds[['amfi_code', 'scheme_name']], on='amfi_code')
    top_fund = fund_pref.sort_values('amount_inr', ascending=False).iloc[0]
    
    cohort_res.append({
        'Cohort Year': cohort,
        'Avg SIP Amount (INR)': avg_sip,
        'Total Invested (INR)': total_invested,
        'Top Fund': top_fund['scheme_name'],
        'Top Fund Invested (INR)': top_fund['amount_inr']
    })

df_cohort_summary = pd.DataFrame(cohort_res)
display(df_cohort_summary)

## 4. SIP Continuity Analysis

To analyze investor discipline and churn, we inspect investors with at least 6 SIP transactions:
- Compute the average gap (in days) between consecutive SIP transaction dates for each investor.
- Flag investors with an average gap greater than 35 days as `'at-risk'` (representing irregular contributions), and others as `'active'`.

In [5]:
df_sip = df_tx[df_tx['transaction_type'] == 'SIP'].sort_values(['investor_id', 'transaction_date'])

sip_gaps = []
for inv, group in df_sip.groupby('investor_id'):
    if len(group) >= 6:
        gaps = group['transaction_date'].diff().dt.days.dropna()
        avg_gap = gaps.mean()
        sip_gaps.append({
            'investor_id': inv,
            'num_sip_tx': len(group),
            'avg_gap': avg_gap,
            'status': 'at-risk' if avg_gap > 35 else 'active'
        })
df_sip_continuity = pd.DataFrame(sip_gaps)

print(f"Total investors with 6+ SIPs: {len(df_sip_continuity)}")
print(f"Active SIP Investors: {(df_sip_continuity['status'] == 'active').sum()}")
print(f"At-Risk SIP Investors: {(df_sip_continuity['status'] == 'at-risk').sum()}")
print(f"Continuity Risk Rate (At-Risk / Total): {(df_sip_continuity['status'] == 'at-risk').mean() * 100:.2f}%")

print("\nFirst 5 SIP Continuity records:")
display(df_sip_continuity.head(5))

## 5. Fund Recommender Engine

A simple rule-based engine mapping risk profiles to matching database risk grades and outputting the top 3 funds ranked by their historical Sharpe ratio.

In [6]:
def recommend_funds(risk_appetite):
    if risk_appetite.lower() == 'low':
        grades = ['Low']
    elif risk_appetite.lower() == 'moderate':
        grades = ['Moderate', 'Moderately High']
    elif risk_appetite.lower() == 'high':
        grades = ['High', 'Very High']
    else:
        return "Invalid appetite. Pick from Low, Moderate, High."
        
    filtered = df_performance[df_performance['risk_grade'].isin(grades)]
    top_3 = filtered.sort_values('sharpe_ratio', ascending=False).head(3)
    return top_3[['scheme_name', 'category', 'risk_grade', 'sharpe_ratio', 'return_3yr_pct']]

print("Recommendations for Moderate Risk Appetite:")
display(recommend_funds('moderate'))

## 6. Sector Concentration Analysis (Herfindahl-Hirschman Index - HHI)

The Herfindahl-Hirschman Index (HHI) measures portfolio concentration. For each equity scheme, we:
1. Group holdings by sector to calculate sector weights $w_s$ (as percentages, 0-100).
2. Sum the squared sector weights: $HHI = \sum (w_s)^2$.
3. Compare concentrations. High HHI ($>2500$) indicates high concentration (e.g. focused sector bets), whereas low HHI ($<1500$) indicates high diversification.

In [7]:
df_eq_holdings = df_holdings.merge(df_funds[['amfi_code', 'category']], on='amfi_code')
df_eq_holdings = df_eq_holdings[df_eq_holdings['category'] == 'Equity']

hhi_records = []
for amfi, group in df_eq_holdings.groupby('amfi_code'):
    name = df_funds[df_funds['amfi_code'] == amfi]['scheme_name'].values[0]
    sector_weights = group.groupby('sector')['weight_pct'].sum()
    hhi_val = (sector_weights ** 2).sum()
    hhi_records.append({
        'amfi_code': amfi,
        'scheme_name': name,
        'sector_hhi': hhi_val,
        'num_sectors': len(sector_weights)
    })

df_hhi = pd.DataFrame(hhi_records).sort_values('sector_hhi', ascending=False)

print("Top 5 Most Concentrated Funds (Highest HHI):")
display(df_hhi.head(5))

print("\nTop 5 Most Diversified Funds (Lowest HHI):")
display(df_hhi.tail(5))

## 7. Advanced Insights & Takeaways

Based on the mathematical and behavioral analyses performed, we establish five key insights:

### 1. Risk Segmentation (VaR & CVaR)
Small Cap schemes represent the highest potential risk. **SBI Small Cap Fund (Direct & Regular)** and **Axis Small Cap Fund** exhibit the highest 95% Historical VaR (up to **2.69% daily expected loss** under normal market stress). Conversely, liquid funds such as **ICICI Pru Liquid Fund** and **ABSL Liquid Fund** maintain an extremely low 95% VaR (less than **0.03% daily loss**), establishing a stark contrast in short-term drawdowns.

### 2. Investor Cohort Trends
The platform experienced massive growth in **2024**, with **4,803 new investors** contributing a total of **2.258 Billion INR (~225.8 Cr)**. In contrast, the **2025 cohort** is smaller (only **197 investors** with **18.99 Million INR** total invested) because the data coverage terminates in May 2025. Crucially, the **2025 cohort has a higher average SIP amount (13,505 INR)** than the **2024 cohort (10,997 INR)**, which suggests that newer investors are committing larger average periodic amounts.

### 3. Systematic Fund Preference
Both cohorts demonstrate a strong preference for Axis Mutual Fund schemes. The **2024 cohort** invested the largest single amount (**65.02 Million INR**) in **Axis Small Cap Fund - Regular - Growth**. The **2025 cohort** preferred **Axis Midcap Fund - Regular - Growth**, putting in **1.28 Million INR**. This indicates that small-and-mid-cap equity options managed by Axis AMC are the primary client acquisition vehicles.

### 4. Severe SIP Continuity Churn Risk
Of the 1,362 investors with 6 or more SIP transactions, **97.8% (1,332 investors)** have an average transaction gap **exceeding 35 days** and are classified as **at-risk**. Since standard SIPs are scheduled monthly (~30-day frequency), this wide average gap indicates highly irregular transaction schedules, missed payments, or execution delays, revealing a high churn risk.

### 5. Portfolio Sector Concentrations (HHI)
HHI calculations reveal a high level of concentration in bluechip funds, led by **Axis Bluechip Fund (HHI = 2967.69)**. This fund places heavy sector bets on Banking/Financials and IT, leaving it vulnerable to sector-specific shocks. On the other hand, mid-and-small-cap schemes like **UTI Mid Cap Fund (HHI = 1240.20)** are much more diversified across a broader spectrum of sectors, maintaining a balanced risk profile.